# Phase 3: Data Preparation
## Parsing and Normalizing the Close File HC Tabs

**CRISP-DM Purpose:** Transform raw data into clean, analysis-ready datasets. Build reusable extraction code that handles the structural quirks discovered in Phase 2.

**Project:** ML-Driven FP&A Analyst -- Headcount Actuals vs. Forecast Analysis

**Scope:** Build `src/parser.py` to extract all 5 HC tabs into normalized DataFrames, validate against known values, and save interim datasets for Phase 4 analysis.

---

## 1. Parser Design Decisions

The Phase 2 data profiling revealed several structural quirks that the parser must handle:

| Challenge | Solution |
|-----------|----------|
| Summary rows at different positions across tabs (166-169 vs 170-173) | Read by **column B label**, not row number |
| 70 always-zero rows (`(Only)` placeholders, COR Only sub-rows) | `filter_analytical()` removes them post-parse |
| Leasing Center US uses hours-based FTE (decimal) while India LC is integer | Tag `hc_type` based on label `Leasing Center - US` specifically |
| Commentary columns contain Excel integer 0 instead of empty | Clean to `None` unless value is a non-empty string |
| Month headers like `Mar-2026` need ISO conversion | `_parse_month()` converts to `YYYY-MM` datetime |
| 5 tabs represent different entity views of the same data | Map each tab to its entity via `TAB_TO_ENTITY` dict |

### Parser Architecture

```
src/parser.py
  ├── load_workbook()         Load Close File with openpyxl (data_only=True)
  ├── parse_actuals()         Extract trailing 12-month grid (cols C-N) → long-format DataFrame
  ├── parse_variance()        Extract month + EOY variance sections → wide-format DataFrame
  └── filter_analytical()     Remove zero-signal rows from either DataFrame
```

### Row Classification

Each department row is classified by the `_classify_level()` function:

| Level | Rule | Example |
|-------|------|---------|
| `L0` | Label == `Entrata` | Grand total |
| `L1` | Label in cost categories (COR, S&M, R&D, G&A) | Cost category subtotal |
| `L2` | Base label found in department hierarchy | `Engineering`, `Sales` |
| `L3` | Contains ` - ` separator | `Engineering - US`, `Sales - India` |
| `only` | Ends with `(Only)` | Filtered out |
| `summary` | Label in summary row list | `Total OPEX`, `Total Entrata`, `HC excl. LC` |

## 2. Output Datasets

The parser produces two primary DataFrames, each available in full and filtered views.

### Actuals DataFrame (`parse_actuals`)

Long-format: one row per department × entity × month.

| Column | Type | Description |
|--------|------|-------------|
| `tab` | str | Source tab name (HC, HC-US, etc.) |
| `entity` | str | Total, US, India, RD, or Colleen |
| `department` | str | Row label from column B |
| `cost_category` | str | L1 parent (COR, S&M, R&D, G&A) |
| `level` | str | Hierarchy level (L0, L1, L2, L3, summary) |
| `hc_type` | str | `fte` for Leasing Center - US, `integer` for all others |
| `month` | datetime | Month of the actuals value |
| `value` | float | Headcount value |

### Variance DataFrame (`parse_variance`)

Wide-format: one row per department × entity, with both month and EOY variance.

| Column | Type | Description |
|--------|------|-------------|
| `tab` | str | Source tab name |
| `entity` | str | Total, US, India, RD, or Colleen |
| `department` | str | Row label from column B |
| `cost_category` | str | L1 parent |
| `level` | str | Hierarchy level |
| `hc_type` | str | `fte` or `integer` |
| `variance_month` | datetime | The month being compared (e.g., 2026-03) |
| `actuals` | float | CWM actuals for the month (column P) |
| `forecast` | float | Latest Forecast for the month (column Q) |
| `month_variance` | float | Actuals - Forecast (column R) |
| `month_commentary` | str/None | Analyst explanation of month variance (column T) |
| `cwm_fy` | float | CWM full-year value (column V) |
| `latest_forecast_fy` | float | Latest Forecast full-year value (column W) |
| `eoy_variance` | float | CWM FY - Latest Forecast FY (column X) |
| `eoy_commentary` | str/None | Analyst explanation of EOY variance (column Z) |

In [ ]:
import sys
sys.path.insert(0, "..")

from src.parser import load_workbook, parse_actuals, parse_variance, filter_analytical

wb = load_workbook()

df_act_raw = parse_actuals(wb)
df_var_raw = parse_variance(wb)

df_act = filter_analytical(df_act_raw)
df_var = filter_analytical(df_var_raw)

print(f"Actuals:  {df_act_raw.shape[0]:,} raw -> {df_act.shape[0]:,} filtered ({df_act_raw.shape[0] - df_act.shape[0]:,} removed)")
print(f"Variance: {df_var_raw.shape[0]:,} raw -> {df_var.shape[0]:,} filtered ({df_var_raw.shape[0] - df_var.shape[0]:,} removed)")

## 3. Validation Against Phase 2 Known Values

Five validation checks confirm the parser reproduces the exact values we profiled manually in Phase 2.

In [ ]:
import pandas as pd

# VALIDATION 1: Entity rollup (Total = US + India + RD + Colleen)
print("VALIDATION 1: Entity rollup for Feb-2026 (Entrata total)")
print("=" * 60)

feb_entrata = df_act[(df_act['department'] == 'Entrata') & (df_act['month'] == '2026-02-01')]
total_val = feb_entrata[feb_entrata['entity'] == 'Total']['value'].values[0]
entity_sum = feb_entrata[feb_entrata['entity'].isin(['US', 'India', 'RD', 'Colleen'])]['value'].sum()
print(f"  Total tab: {total_val}")
print(f"  Entity sum (US+India+RD+Colleen): {entity_sum}")
print(f"  Result: {'PASS' if abs(total_val - entity_sum) < 0.01 else 'FAIL'}")

# VALIDATION 2: Month variance formula
print(f"\nVALIDATION 2: Month variance = Actuals - Forecast")
print("=" * 60)

var_total = df_var[(df_var['entity'] == 'Total') & (df_var['level'].isin(['L0', 'L1']))]
for _, row in var_total.iterrows():
    expected = round(row['actuals'] - row['forecast'], 2)
    actual = round(row['month_variance'], 2)
    print(f"  {row['department']}: {row['actuals']} - {row['forecast']} = {actual} [{'PASS' if abs(expected - actual) < 0.1 else 'FAIL'}]")

# VALIDATION 3: EOY variance formula
print(f"\nVALIDATION 3: EOY variance = CWM FY - Latest Forecast FY")
print("=" * 60)

for _, row in var_total.iterrows():
    expected = round(row['cwm_fy'] - row['latest_forecast_fy'], 2)
    actual = round(row['eoy_variance'], 2)
    print(f"  {row['department']}: {row['cwm_fy']} - {row['latest_forecast_fy']} = {actual} [{'PASS' if abs(expected - actual) < 0.1 else 'FAIL'}]")

In [ ]:
# VALIDATION 4: Spot-check known values from Phase 2
print("VALIDATION 4: Spot-check against Phase 2 profiled values")
print("=" * 60)

checks = [
    ("Total", "Entrata", "2026-02", 2128.5),
    ("Total", "COR", "2026-02", 795.5),
    ("Total", "Engineering", "2026-02", 717),
    ("US", "Engineering - US", "2026-02", 92),
    ("India", "Engineering - India", "2026-02", 584),
    ("Total", "Leasing Center", "2026-02", 151.5),
    ("US", "Leasing Center - US", "2026-02", 108.5),
    ("India", "Leasing Center - India", "2026-02", 43),
]

for entity, dept, month_str, expected in checks:
    mask = (
        (df_act['entity'] == entity)
        & (df_act['department'] == dept)
        & (df_act['month'] == f"{month_str}-01")
    )
    actual = df_act[mask]['value'].values
    status = f"PASS ({actual[0]})" if len(actual) == 1 and abs(actual[0] - expected) < 0.01 else f"FAIL ({actual})"
    print(f"  {entity}/{dept} {month_str}: expected={expected} [{status}]")

# VALIDATION 5: Commentary preservation
print(f"\nVALIDATION 5: Commentary rows preserved")
print("=" * 60)

month_comm = df_var[df_var['month_commentary'].notna()]
eoy_comm = df_var[df_var['eoy_commentary'].notna()]
print(f"  Month commentary rows: {len(month_comm)}")
print(f"  EOY commentary rows: {len(eoy_comm)}")

for _, row in month_comm.iterrows():
    print(f"  [{row['entity']}] {row['department']}: var={row['month_variance']}, '{row['month_commentary'][:70]}...'")

## 4. Filtered Dataset Profile

In [ ]:
print("ACTUALS DATASET PROFILE")
print("=" * 60)
print(f"Shape: {df_act.shape}")
print(f"\nEntities: {sorted(df_act['entity'].unique())}")
print(f"Levels: {sorted(df_act['level'].unique())}")
print(f"HC Types: {sorted(df_act['hc_type'].unique())}")
print(f"Months: {df_act['month'].min().strftime('%Y-%m')} to {df_act['month'].max().strftime('%Y-%m')} ({df_act['month'].nunique()} months)")
print(f"Departments: {df_act['department'].nunique()} unique")

print(f"\nRows by level:")
print(df_act.groupby('level').size().to_string())

print(f"\nRows by entity:")
print(df_act.groupby('entity').size().to_string())

print(f"\nFTE-tagged rows: {len(df_act[df_act['hc_type'] == 'fte'])} (all Leasing Center - US)")

print(f"\n\nVARIANCE DATASET PROFILE")
print("=" * 60)
print(f"Shape: {df_var.shape}")
print(f"Variance month: {df_var['variance_month'].iloc[0].strftime('%Y-%m')}")
print(f"\nMonth commentary: {df_var['month_commentary'].notna().sum()} rows with text")
print(f"EOY commentary: {df_var['eoy_commentary'].notna().sum()} rows with text")

## 5. Interim Datasets Saved

Six parquet files are saved to `data/interim/` for use in Phase 4:

| File | Description | Rows | Use Case |
|------|-------------|------|----------|
| `actuals.parquet` | All entities, all levels | 7,548 | Full analytical dataset |
| `variance.parquet` | All entities, all levels | 629 | Full variance dataset |
| `actuals_total.parquet` | Total entity only | 1,488 | Company-wide trend analysis |
| `variance_total.parquet` | Total entity only | 124 | Company-wide variance review |
| `actuals_by_entity.parquet` | US/India/RD/Colleen (no Total) | 6,060 | Entity-level decomposition |
| `variance_by_entity.parquet` | US/India/RD/Colleen (no Total) | 505 | Entity-level variance drill-down |

### Why Parquet?

- Preserves column types (datetime, float, string) without CSV ambiguity
- Smaller file size than CSV for numeric-heavy data
- Fast read times for pandas (`pd.read_parquet()`)
- Parquet files are in `.gitignore` (derived from the Close File, not source data)

## 6. Transformation Summary

### What Changed from Raw to Analytical

| Step | Before | After |
|------|--------|-------|
| **Structure** | 5 separate Excel tabs, each 175 rows × 26 cols | 2 normalized DataFrames (actuals + variance) |
| **Row count** | 9,864 raw actuals rows | 7,548 analytical rows (2,316 removed) |
| **Zero rows** | 70 always-zero `(Only)` and COR Only rows per tab | Filtered out |
| **Entity mapping** | Tab name implies entity | Explicit `entity` column (Total, US, India, RD, Colleen) |
| **Level hierarchy** | Implicit from label naming | Explicit `level` column (L0-L3, summary) |
| **Cost category** | Implicit from row position | Explicit `cost_category` column (COR, S&M, R&D, G&A) |
| **HC type** | Implicit (only LC-US is decimal) | Explicit `hc_type` column (fte vs integer) |
| **Commentary** | Excel integer 0 mixed with text | Cleaned to `None` or string only |
| **Months** | `Mar-2026` string labels | ISO datetime objects |

### Data Lineage

```
Close File - Master.xlsx (raw, on desktop)
  └─ src/parser.py
      ├─ parse_actuals()  →  data/interim/actuals.parquet
      │                      data/interim/actuals_total.parquet
      │                      data/interim/actuals_by_entity.parquet
      ├─ parse_variance() →  data/interim/variance.parquet
      │                      data/interim/variance_total.parquet
      │                      data/interim/variance_by_entity.parquet
      └─ filter_analytical() applied to both
```

## 7. Phase 3 Status

| Deliverable | Status |
|-------------|--------|
| `src/parser.py` built with label-based reading | Done |
| Actuals DataFrame (long-format, 12 months × 5 entities × 127 departments) | Done |
| Variance DataFrame (month + EOY, with commentary) | Done |
| Zero-row filtering (`filter_analytical`) | Done |
| FTE tagging for Leasing Center - US | Done |
| Commentary cleaning (integer 0 → None) | Done |
| 5 validation checks against Phase 2 values | All pass |
| Interim parquet files saved | Done |

**Next step:** Phase 4 -- Analysis (to be initiated upon manager approval)

Phase 4 will use these prepared datasets to:
- Decompose the monthly variance by cost category, department, and entity
- Identify the top variance drivers for the March close
- Analyze EOY forecast trajectory and departments trending off-plan
- Surface patterns in commentary (swizzle frequency, capacity hire changes, etc.)